# Week 89, Day 4: Tests

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w089d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('waymark_domain.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\n\n\nclass UnknownPlace(LookupError):\n    pass\n\n\nclass NoRoute(Exception):\n    pass\n\n\ndef naismith_minutes(km, climb_m):\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n\n\nclass Route:\n    def __init__(self, stops, km, climb_m):\n        self.stops, self.km, self.climb_m = list(stops), km, climb_m\n\n    @property\n    def minutes(self):\n        return naismith_minutes(self.km, self.climb_m)\n\n    def __repr__(self):\n        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"\n\n\nclass PathNetwork:\n    def __init__(self, heights):\n        self.heights = dict(heights)\n        self.paths = {name: {} for name in self.heights}\n\n    def _check(self, *names):\n        for name in names:\n            if name not in self.heights:\n                raise UnknownPlace(name)\n\n    def add_path(self, a, b, km):\n        self._check(a, b)\n        if km <= 0:\n            raise ValueError("km must be positive")\n        self.paths[a][b] = km\n        self.paths[b][a] = km\n\n    def climb(self, stops):\n        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))\n\n    def shortest(self, start, end):\n        self._check(start, end)\n        dist, prev, done = {start: 0}, {}, set()\n        heap = [(0, start)]\n        while heap:\n            d, node = heapq.heappop(heap)\n            if node in done:\n                continue\n            done.add(node)\n            if node == end:\n                break\n            for nxt, km in sorted(self.paths[node].items()):\n                if d + km < dist.get(nxt, float("inf")):\n                    dist[nxt] = d + km\n                    prev[nxt] = node\n                    heapq.heappush(heap, (d + km, nxt))\n        if end not in done:\n            raise NoRoute(f"no route from {start} to {end}")\n        stops = [end]\n        while stops[-1] != start:\n            stops.append(prev[stops[-1]])\n        stops.reverse()\n        return Route(stops, round(dist[end], 2), self.climb(stops))\n')
with open('waymark_map.py', 'w', encoding='utf-8') as f:
    f.write('from waymark_domain import PathNetwork\n\nHEIGHTS = {"Mill": 40, "Weir": 55, "Church": 80, "Ridge": 310, "Farm": 120, "Wood": 150, "Lake": 60, "Island": 5}\nPATHS = [("Mill", "Weir", 1.5), ("Weir", "Church", 2.0), ("Mill", "Church", 4.0), ("Church", "Ridge", 3.0),\n         ("Weir", "Farm", 2.5), ("Farm", "Ridge", 2.0), ("Farm", "Wood", 1.5), ("Wood", "Lake", 2.0), ("Lake", "Mill", 3.5)]\n\n\ndef sample_network():\n    network = PathNetwork(HEIGHTS)\n    for a, b, km in PATHS:\n        network.add_path(a, b, km)\n    return network\n')
with open('waymark_geo.py', 'w', encoding='utf-8') as f:
    f.write('import math\nfrom waymark_domain import PathNetwork\nfrom waymark_map import HEIGHTS, PATHS\n\nPOSITIONS = {"Mill": (0.0, 0.0), "Weir": (1.4, 0.4), "Church": (2.8, 1.2), "Ridge": (4.4, 3.4), "Farm": (2.8, 2.4),\n             "Wood": (2.0, 3.5), "Lake": (0.6, 2.8), "Island": (-3.0, 5.0)}\n\n\nclass GeoNetwork(PathNetwork):\n    """A path network whose junctions also have positions, in kilometres east and north."""\n\n    def __init__(self, heights, positions):\n        super().__init__(heights)\n        self.positions = dict(positions)\n\n    def straight_km(self, a, b):\n        return math.dist(self.positions[a], self.positions[b])\n\n\ndef sample_geo():\n    network = GeoNetwork(HEIGHTS, POSITIONS)\n    for a, b, km in PATHS:\n        network.add_path(a, b, km)\n    return network\n\n\ndef grid_network(size):\n    """A size-by-size grid of junctions named \'r-c\', 1 km apart, with slightly longer paths to their neighbours."""\n    names = {(r, c): f"{r}-{c}" for r in range(size) for c in range(size)}\n    heights = {name: ((r * 13 + c * 7) % 9) * 20 for (r, c), name in names.items()}\n    positions = {name: (float(c), float(r)) for (r, c), name in names.items()}\n    network = GeoNetwork(heights, positions)\n    for (r, c), name in names.items():\n        if c + 1 < size:\n            network.add_path(name, names[(r, c + 1)], 1.0 + ((r * 7 + c * 3) % 5) / 10)\n        if r + 1 < size:\n            network.add_path(name, names[(r + 1, c)], 1.0 + ((r * 3 + c * 5) % 5) / 10)\n    return network\n')
with open('waymark_kinds.py', 'w', encoding='utf-8') as f:
    f.write('KINDS = {\n    frozenset({"Mill", "Church"}): "road",\n    frozenset({"Lake", "Mill"}): "road",\n    frozenset({"Weir", "Farm"}): "track",\n    frozenset({"Wood", "Lake"}): "track",\n}\n\n\ndef kind_of(a, b):\n    """The kind of the path between two junctions: road, track, or (by default) path."""\n    return KINDS.get(frozenset({a, b}), "path")\n')
with open('waymark_prefs.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\nfrom dataclasses import dataclass\nfrom waymark_domain import NoRoute, Route, UnknownPlace\nfrom waymark_kinds import kind_of\nfrom waymark_map import sample_network\n\n@dataclass(frozen=True)\nclass Preferences:\n    avoid_roads: bool = False\n    avoid_steep: bool = False\n    road_factor: float = 3.0\n    steep_factor: float = 2.0\n    steep_limit: int = 80\n\n    def __post_init__(self):\n        if self.road_factor < 1 or self.steep_factor < 1:\n            raise ValueError("factors must be at least 1")\n        if not self.steep_limit > 0:\n            raise ValueError("steep_limit must be positive")\n\n    @classmethod\n    def from_query(cls, query):\n        avoid = {a.strip() for a in query.get("avoid", "").split(",") if a.strip()}\n        for item in sorted(avoid):\n            if item not in ("roads", "steep"):\n                raise ValueError(f"cannot avoid {item}")\n        options = {"avoid_roads": "roads" in avoid, "avoid_steep": "steep" in avoid}\n        if "steep_limit" in query:\n            try:\n                options["steep_limit"] = int(query["steep_limit"])\n            except ValueError:\n                raise ValueError("steep_limit must be a whole number")\n        return cls(**options)\n\ndef step_cost(network, a, b, prefs):\n    km = network.paths[a][b]\n    cost = km * (prefs.road_factor if prefs.avoid_roads and kind_of(a, b) == "road" else 1)\n    climb = max(0, network.heights[b] - network.heights[a])\n    return cost * (prefs.steep_factor if prefs.avoid_steep and climb / km > prefs.steep_limit else 1)\n\ndef best_route(network, start, end, prefs):\n    best, prev, done, heap = {start: 0}, {}, set(), [(0, start)]\n    while heap:\n        c, node = heapq.heappop(heap)\n        if node in done:\n            continue\n        done.add(node)\n        if node == end:\n            stops = [end]\n            while stops[-1] != start:\n                stops.append(prev[stops[-1]])\n            stops.reverse()\n            km = sum(network.paths[a][b] for a, b in zip(stops, stops[1:]))\n            return Route(stops, round(km, 2), network.climb(stops)), round(c, 2)\n        for nxt in sorted(network.paths[node]):\n            step = step_cost(network, node, nxt, prefs)\n            if c + step < best.get(nxt, float("inf")):\n                best[nxt], prev[nxt] = c + step, node\n                heapq.heappush(heap, (c + step, nxt))\n    raise NoRoute(f"no route from {start} to {end}")\n\ndef explain(network, stops, shortest_km, steep_limit=80):\n    steps = list(zip(stops, stops[1:]))\n    out = []\n    road = sum(network.paths[a][b] for a, b in steps if kind_of(a, b) == "road")\n    if road:\n        out.append(f"{round(road, 1)} km on roads")\n    steep = [f"{a} to {b}" for a, b in steps if max(0, network.heights[b] - network.heights[a]) / network.paths[a][b] > steep_limit]\n    out.append(f"{len(steep)} steep climb{\'s\' if len(steep) > 1 else \'\'}: {\', \'.join(steep)}" if steep else "no steep climbs")\n    km = sum(network.paths[a][b] for a, b in steps)\n    if km - shortest_km > 0.05:\n        out.append(f"{round(km - shortest_km, 1)} km longer than the shortest route")\n    return out\n\nCHOICES = [("shortest", Preferences()), ("flattest", Preferences(avoid_steep=True)), ("quietest", Preferences(avoid_roads=True))]\n\ndef alternatives(network, start, end):\n    found = {}\n    for label, prefs in CHOICES:\n        route, _ = best_route(network, start, end, prefs)\n        key = tuple(route.stops)\n        if key in found:\n            found[key]["labels"].append(label)\n        else:\n            found[key] = {"labels": [label], "route": route}\n    shortest_km = next(iter(found.values()))["route"].km\n    return [{"label": " and ".join(v["labels"]), "stops": v["route"].stops, "km": round(v["route"].km, 1),\n             "minutes": v["route"].minutes, "why": explain(network, v["route"].stops, shortest_km)}\n            for v in found.values()]\n')
with open('loops.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\nfrom waymark_domain import Route\n\n\ndef shortest_avoiding(network, start, end, banned):\n    dist, prev, done, heap = {start: 0}, {}, set(), [(0, start)]\n    while heap:\n        d, node = heapq.heappop(heap)\n        if node in done:\n            continue\n        done.add(node)\n        if node == end:\n            stops = [end]\n            while stops[-1] != start:\n                stops.append(prev[stops[-1]])\n            stops.reverse()\n            return Route(stops, round(d, 2), network.climb(stops))\n        for nxt, km in sorted(network.paths[node].items()):\n            if frozenset({node, nxt}) in banned:\n                continue\n            if d + km < dist.get(nxt, float("inf")):\n                dist[nxt], prev[nxt] = d + km, node\n                heapq.heappush(heap, (d + km, nxt))\n    return None\n\n\ndef paths_of(stops):\n    return {frozenset(pair) for pair in zip(stops, stops[1:])}\n\n\ndef loop_via(network, start, turn):\n    if turn == start:\n        return None\n    out = shortest_avoiding(network, start, turn, set())\n    if out is None:\n        return None\n    back = shortest_avoiding(network, turn, start, paths_of(out.stops))\n    if back is None:\n        return None\n    stops = out.stops + back.stops[1:]\n    return Route(stops, round(out.km + back.km, 2), network.climb(stops))\n')
with open('routes.json', 'w', encoding='utf-8') as f:
    f.write('[\n  {"name": "S1 shortest to the ridge", "from": "Mill", "to": "Ridge", "avoid": "", "stops": ["Mill", "Weir", "Farm", "Ridge"]},\n  {"name": "S9 flattest to the ridge", "from": "Mill", "to": "Ridge", "avoid": "steep", "stops": ["Mill", "Weir", "Church", "Ridge"]},\n  {"name": "S9 quietest to the wood", "from": "Mill", "to": "Wood", "avoid": "roads", "stops": ["Mill", "Weir", "Farm", "Wood"]},\n  {"name": "S1 downhill home", "from": "Ridge", "to": "Mill", "avoid": "", "stops": ["Ridge", "Farm", "Weir", "Mill"]}\n]\n')

**Predict before you run:** what will this print?



In [ ]:
import itertools
from waymark_prefs import Preferences

combos = list(itertools.product([False, True], [False, True], [60, 80, 120]))
print(len(combos), combos[:3])
prefs = [Preferences(avoid_roads=r, avoid_steep=s, steep_limit=l) for r, s, l in combos]
print(len(set(prefs)), prefs[-1])

> **Key idea.** Test performance by how counted work grows as input size doubles (ratio near 2 is linear, near 4 quadratic), not by elapsed time. Generate all combinations of options with `itertools.product` and check invariants for each: right ends, cost at least distance, nothing cheaper under the same preferences. Write metamorphic tests where the right answer is unknown but its change is: adding paths never lengthens, scaling lengths scales distances, reversing keeps distances. Record checked results in a regression suite named by story, replay it after every change, and accept differences deliberately.

### Exercise: How does the work grow?

Write `growth_table(sizes, measure)`: for each size, `measure(size)` gives a count of work; return a list of `(size, work, ratio)` where `ratio` is the work divided by the previous row's work, rounded to 2 places (`None` for the first row). Then `classify(table)`: using the ratios after the first row, return `"linear"` if every ratio is at most 2.5, `"quadratic"` if every ratio is at most 5, else `"worse than quadratic"`.

In [ ]:
def growth_table(sizes, measure):
    ___

def classify(table):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_growth():\n    \"ratios when the size doubles\"\n    t = growth_table([10, 20, 40], lambda n: 3 * n + 4)\n    assert t == [(10, 34, None), (20, 64, 1.88), (40, 124, 1.94)] and classify(t) == \"linear\"\n    assert classify(growth_table([10, 20, 40], lambda n: n * n)) == \"quadratic\"\n    assert classify(growth_table([4, 8, 16], lambda n: 2 ** n)) == \"worse than quadratic\"\n    from waymark_geo import grid_network\n    assert classify(growth_table([4, 8, 16], lambda n: len(grid_network(n).heights))) == \"quadratic\", \"a grid's junctions grow with the square of its side\"")

### Exercise: Every combination, checked

Using the page's `waymark_prefs`, write `all_preferences()`: the 12 `Preferences` from every combination of `avoid_roads` in `[False, True]`, `avoid_steep` in `[False, True]` and `steep_limit` in `[60, 80, 120]` (in `itertools.product` order). Then `invariant_failures(network, start, end, prefs)`: a list of failure messages for the best route under `prefs`: `"wrong ends"` unless it starts at `start` and ends at `end`; `"cost below distance"` if its cost is less than its km (allowing 0.001); `"km mismatch"` if its km differs from the sum of its paths by more than 0.01; and `"shortest is cheaper"` if the plain shortest route's cost under the same preferences (the sum of `step_cost` over its steps) is lower by more than 0.001.

In [ ]:
import itertools
from waymark_prefs import Preferences, best_route, step_cost

def all_preferences():
    ___

def invariant_failures(network, start, end, prefs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from waymark_map import sample_network\nfrom waymark_geo import grid_network\n\ndef test_matrix():\n    \"twelve combinations, every pair\"\n    prefs = all_preferences()\n    assert len(prefs) == 12 and len(set(prefs)) == 12 and prefs[0] == Preferences(steep_limit=60)\n    net = sample_network()\n    names = [n for n in net.heights if n != \"Island\"]\n    for p in prefs:\n        for a in names:\n            for b in names:\n                if a != b:\n                    assert invariant_failures(net, a, b, p) == [], (a, b, p)\n    grid = grid_network(6)\n    assert invariant_failures(grid, \"0-0\", \"5-5\", prefs[-1]) == []\n\ndef test_catches():\n    \"a broken cost is noticed\"\n    import waymark_prefs\n    original = waymark_prefs.step_cost\n    waymark_prefs.step_cost = lambda network, a, b, prefs: network.paths[a][b] * 0.5\n    try:\n        failures = invariant_failures(sample_network(), \"Mill\", \"Ridge\", Preferences())\n    finally:\n        waymark_prefs.step_cost = original\n    assert \"cost below distance\" in failures")

### Exercise: Metamorphic relations

Write `relation_failures(make_network, pairs, shortest)`: `make_network()` builds a fresh network; `shortest(network, a, b)` returns a route's km. Check three relations for every pair `(a, b)` and return a list of failures as `"RELATION: a to b"`: `"symmetric"` (the distance from `a` to `b` equals `b` to `a`, within 0.001); `"scaling"` (on a network where every path is doubled, the distance is double, within 0.001); `"adding"` (after adding a path of 0.1 km between `a` and `b` when they aren't already joined, the distance is at most 0.1). Build a fresh network for each changed version.

In [ ]:
def relation_failures(make_network, pairs, shortest):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from waymark_map import sample_network\n\ndef real(network, a, b):\n    return network.shortest(a, b).km\n\ndef test_relations():\n    \"no expected answers needed\"\n    pairs = [(\"Mill\", \"Ridge\"), (\"Lake\", \"Church\"), (\"Wood\", \"Weir\")]\n    assert relation_failures(sample_network, pairs, real) == []\n\ndef test_catches():\n    \"a broken search breaks a relation\"\n    def nearest_first(network, a, b):\n        km, node, seen = 0, a, {a}\n        while node != b:\n            options = [n for n in network.paths[node] if n not in seen]\n            nxt = b if b in network.paths[node] else min(options, key=lambda n: (network.paths[node][n], n))\n            km, node = km + network.paths[node][nxt], nxt\n            seen.add(node)\n        return round(km, 2)\n    failures = relation_failures(sample_network, [(\"Mill\", \"Ridge\"), (\"Lake\", \"Church\")], nearest_first)\n    assert \"symmetric: Mill to Ridge\" in failures")

### Exercise: Tests for the round walks

`loops.py` (a page file) has Day 2's `shortest_avoiding`, `paths_of` and `loop_via`. Write pytest tests that pass on it and catch all four broken versions. Build the sample network with `waymark_map.sample_network()`.

In [ ]:
import pytest
from loops import loop_via, paths_of, shortest_avoiding
from waymark_map import sample_network

def test_loop_exists():
    assert loop_via(sample_network(), "Mill", "Farm") is not None

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"loops.py\").read()\nBROKEN = {\n    \"bans one direction\": GOOD.replace(\"frozenset({node, nxt}) in banned\", \"(node, nxt) in banned\"),\n    \"turning at the start\": GOOD.replace(\"    if turn == start:\\n        return None\\n\", \"\"),\n    \"only the outward km\": GOOD.replace(\"round(out.km + back.km, 2)\", \"round(out.km, 2)\"),\n    \"turn twice\": GOOD.replace(\"out.stops + back.stops[1:]\", \"out.stops + back.stops\"),\n}\ndef test_suite():\n    \"all four caught\"\n    for name, code in BROKEN.items():\n        assert code != GOOD, name\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"loops\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"loops.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Bug hunt: Tripled, but not equal

The scaling relation compares floats exactly, so three times a distance against the distance on a map with every path tripled fails on rounding: it prints `failures: 1`. Compare within a small tolerance, so it prints `failures: 0`.

In [ ]:
d = 0.1 + 0.2
tripled = 0.3 + 0.6
failures = 0 if tripled == 3 * d else 1
print("failures:", failures)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"tolerance\"\n    expect_output(\"failures: 0\")")

### Bug hunt: Twelve combinations, or one?

The combinations are built by nesting loops that overwrite one dict, so every entry in the list is the same object, the last combination: it prints `distinct: 1`. Build a new dict each time (or use `itertools.product`), so it prints `distinct: 12`.

In [ ]:
combos = []
options = {}
for roads in (False, True):
    for steep in (False, True):
        for limit in (60, 80, 120):
            options["roads"], options["steep"], options["limit"] = roads, steep, limit
            combos.append(options)
print("distinct:", len({tuple(sorted(c.items())) for c in combos}))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a new value each time\"\n    expect_output(\"distinct: 12\")")

### Puzzle: A metamorphic test

Arrange a pytest test of the scaling relation. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the number of combinations of 2 booleans and a list of 4 limits
q = ___   # "linear" or "quadratic": work going 100, 410, 1600 as the size doubles
r = ___   # True or False: adding a path can make a shortest distance longer
s = ___   # the ratio of distances after multiplying every path by 3
t = ___   # "accept on purpose" or "ignore": what to do when a recorded route changes after an improvement

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 16 and q == \"quadratic\" and r is False, \"p, q and r\"\n    assert s == 3 and t == \"accept on purpose\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a recorded regression suite

`routes.json` (a page file) holds recorded cases: a `name`, `from`, `to`, `avoid` (as in a query string) and the expected `stops`. Write `replay(cases, plan)`, where `plan(start, end, avoid)` returns a list of stops: return `(passed, changed)`, where `changed` lists, for each case whose stops differ, `{"name", "expected", "actual"}`. Then `record(cases, plan)`: a new list of cases (copies) with `stops` replaced by what `plan` returns now. And `plan_with_prefs(network)`: a `plan` function using the page's `waymark_prefs` (`Preferences.from_query({"avoid": avoid})` and `best_route`). Finally `report(passed, changed)`: `"4 passed, 0 changed"`, followed, for each change, by a line `"CHANGED name: A > B > C became A > D > C"`.

In [ ]:
import copy
import json
from waymark_prefs import Preferences, best_route

def replay(cases, plan):
    ___

def record(cases, plan):
    ___

def plan_with_prefs(network):
    ___

def report(passed, changed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import json\nfrom waymark_map import sample_network\n\ndef _cases():\n    return json.load(open(\"routes.json\"))\n\ndef test_replay():\n    \"today's answers match the recording\"\n    cases = _cases()\n    passed, changed = replay(cases, plan_with_prefs(sample_network()))\n    assert (passed, changed) == (4, []) and report(passed, changed) == \"4 passed, 0 changed\"\n\ndef test_change_noticed():\n    \"a change is reported, then accepted on purpose\"\n    net = sample_network()\n    net.add_path(\"Mill\", \"Ridge\", 3.3)\n    plan = plan_with_prefs(net)\n    passed, changed = replay(_cases(), plan)\n    assert passed == 2 and [c[\"name\"] for c in changed] == [\"S1 shortest to the ridge\", \"S1 downhill home\"]\n    assert report(passed, changed).splitlines()[1] == \"CHANGED S1 shortest to the ridge: Mill > Weir > Farm > Ridge became Mill > Ridge\"\n    original = _cases()\n    accepted = record(original, plan)\n    assert replay(accepted, plan) == (4, []) and original[0][\"stops\"] == [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"], \"record copies\"")

## Warm-up

Twelve combinations from three small lists: `itertools.product` (Week 59) makes them all, and because `Preferences` is a frozen dataclass they can go in a set. Testing every combination finds the bug that only appears when two options interact.

## Growth, not timing

A performance test that asserts "under 50 milliseconds" fails on a slow machine and passes on a fast one whatever the code does. Test the **shape** instead: run the algorithm on inputs of doubling size, count its work, and look at the ratio between successive sizes. A ratio near 2 when the size doubles means the work grows **linearly**; near 4, **quadratically**; a ratio that itself keeps growing means something worse. A test can assert "doubling the map at most roughly doubles the round-walk searches", and that holds on any machine.

## Every combination

When options can interact, test all their combinations, generated with `itertools.product` (or `pytest.mark.parametrize` stacked, Week 72). Hand-written expected answers for twelve combinations and many routes are impractical, so check **invariants** that hold for every one: the route starts and ends where asked; its cost is at least its distance (factors are at least 1); no other route (in particular, the plain shortest one) is cheaper under the same preferences; and the reported distance equals the sum of the paths.

## Metamorphic tests

Sometimes you can't say what the right answer is, but you can say how it must **change**. Those relations are **metamorphic** tests:

- adding a path can never make any shortest distance **longer**;
- doubling every path's length must exactly double every shortest distance (and leave the routes the same);
- the shortest distance from A to B equals the one from B to A (paths are two-way), though the times may differ because of the climb.

Each relation is checked on many inputs, including random ones, and a failure points straight at a bug in the algorithm.

## Regression suites

When you've checked a route by hand (or it's in a scenario), **record** it: a file of cases (start, end, preferences, expected stops) that a test replays after every change. A changed answer isn't necessarily wrong (a better algorithm may find a better route), but it must be **noticed** and accepted on purpose, like a golden file. Name each case after its story, so a failure says which promise to a user has changed.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **One growth test** for your most expensive operation, on doubling sizes, asserting its class.
2. **A matrix** of your options or inputs, with invariants checked for every combination.
3. **Two metamorphic relations** for your trickiest calculation.
4. **A regression suite** of a few recorded, checked results, replayed in CI.
5. **Merge**, and count your tests again.

## Before you go

Tomorrow is **code review** again, this time for algorithm and performance changes, and for the tests themselves.